# 模块二：SwellEx-96 GPU 网络训练、预测与实验结果分析

本 Notebook 专为 **Google Colab GPU 运行时**（`Runtime -> Change runtime type -> T4 / L4 / A100 GPU`）设计，直接消费 **模块一（`01_data_processing_cpu.ipynb`，基于 Google Drive 文件夹 [`1FfLqpZLlSP_R8mhTgwq5xvIUOJdC71Xj`](https://drive.google.com/drive/folders/1FfLqpZLlSP_R8mhTgwq5xvIUOJdC71Xj?usp=drive_link) 生成）** 预先保存在 `/content/drive/MyDrive/SwellEx_Colab/datasets/` 中的 HDF5 数据集，**无需再次读取 `Origindata/`，也不会重复执行任何 CPU 频域特征提取**。

### 高效 I/O 与持久化策略
1. **自动引导仓库**：Colab 会话启动后先挂载 Google Drive，若实例中尚未拉取仓库则自动克隆 `SwellEx` 到 `/content/SwellEx`。
2. **读数据走本地 NVMe 缓存**：根据 Google Drive 上的 `dataset_manifest.json`，按需仅把当前实验用到的 HDF5 目录复制到 Colab 本地高速盘 `/content/local_datasets/`，避免多轮（100 Epochs）`DataLoader` 反复穿透 Google Drive FUSE 造成 GPU 空转等待。
3. **写结果直存 Google Drive**：模型权重（`best.pt`）、训练日志、测试集预测 CSV 与评估图表直接写入 `/content/drive/MyDrive/SwellEx_Colab/experiments/<run_name>/`，即使 Colab 会话断开也不会丢失成果。

In [ ]:
# Cell 1: Google Drive 挂载、仓库自动引导、GPU 环境检查与 dataset_manifest.json 读取
from pathlib import Path
import importlib.util
import json
import os
import shutil
import subprocess
import sys

# --- 与模块一保持一致的 Google Drive 工作区配置 ---
DRIVE_WORKSPACE_SUBDIR = "SwellEx_Colab"  # 对应 /content/drive/MyDrive/SwellEx_Colab
ORIGINDATA_DRIVE_URL = "https://drive.google.com/drive/folders/1FfLqpZLlSP_R8mhTgwq5xvIUOJdC71Xj?usp=drive_link"
REPO_GIT_URL = "https://github.com/AmazingUmi/SwellEx.git"
REPO_GIT_BRANCH = "main"
AUTO_UPDATE_REPO = True     # Colab 下每次运行 Cell 1 自动向 GitHub 拉取并同步最新提交
FORCE_RECLONE_REPO = False  # 若想彻底删除 /content/SwellEx 重新克隆，设为 True 运行一次即可
# ------------------------------------------------

IN_COLAB = (
    "google.colab" in sys.modules
    or (importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None)
)

DRIVE_MOUNT_ROOT = Path("/content/drive")
if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount(str(DRIVE_MOUNT_ROOT), force_remount=False)


def get_git_commit_info(repo_dir: Path) -> tuple[str, str]:
    proc = subprocess.run(
        ["git", "-C", str(repo_dir), "log", "-1", "--format=%h|%s"],
        capture_output=True,
        text=True,
    )
    if proc.returncode == 0 and "|" in proc.stdout:
        short_sha, subject = proc.stdout.strip().split("|", 1)
        return short_sha, subject
    return "unknown", ""


def reload_project_modules() -> None:
    for mod_name in [m for m in list(sys.modules) if m == "scripts" or m.startswith("scripts.")]:
        sys.modules.pop(mod_name, None)
    importlib.invalidate_caches()


def ensure_project_root() -> Path:
    if not IN_COLAB:
        for candidate in (Path.cwd(), Path.cwd().parent):
            if (candidate / "scripts" / "remote_workflow" / "packs.py").exists():
                return candidate.resolve()
        raise FileNotFoundError("未找到项目根目录（缺少 scripts/remote_workflow/packs.py）。")

    colab_repo = Path("/content/SwellEx")
    if FORCE_RECLONE_REPO and colab_repo.exists():
        shutil.rmtree(colab_repo)
        print(f"🗑️ 已按 FORCE_RECLONE_REPO=True 清除旧目录: {colab_repo}")

    if (colab_repo / ".git").is_dir() and (colab_repo / "scripts" / "remote_workflow" / "packs.py").exists():
        old_sha, _ = get_git_commit_info(colab_repo)
        if AUTO_UPDATE_REPO:
            subprocess.run(["git", "-C", str(colab_repo), "remote", "set-url", "origin", REPO_GIT_URL], capture_output=True)
            fetch_proc = subprocess.run(
                ["git", "-C", str(colab_repo), "fetch", "--depth", "1", "origin", REPO_GIT_BRANCH],
                capture_output=True,
                text=True,
            )
            if fetch_proc.returncode == 0:
                subprocess.run(["git", "-C", str(colab_repo), "reset", "--hard", "FETCH_HEAD"], check=True, capture_output=True)
                subprocess.run(
                    ["git", "-C", str(colab_repo), "clean", "-fd", "-e", "outputs", "-e", "output", "-e", "Origindata"],
                    capture_output=True,
                )
                new_sha, subject = get_git_commit_info(colab_repo)
                if old_sha != new_sha:
                    print(f"🔄 已同步云端仓库最新提交: {old_sha} -> {new_sha} ({subject})")
                else:
                    print(f"✅ 云端仓库已是最新提交: {new_sha} ({subject})")
            else:
                print(f"⚠️ 检查远端更新失败（继续使用现有副本 {old_sha}）: {fetch_proc.stderr.strip()}")
        return colab_repo.resolve()

    if colab_repo.exists():
        shutil.rmtree(colab_repo)
    print(f"📥 正在克隆最新云端仓库 {REPO_GIT_URL} ({REPO_GIT_BRANCH}) -> {colab_repo} ...")
    proc = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_GIT_BRANCH, REPO_GIT_URL, str(colab_repo)],
        capture_output=True,
        text=True,
    )
    if proc.returncode == 0 and (colab_repo / "scripts" / "remote_workflow" / "packs.py").exists():
        new_sha, subject = get_git_commit_info(colab_repo)
        print(f"✅ 克隆完成: {new_sha} ({subject})")
        return colab_repo.resolve()

    fallback_candidates = [Path.cwd(), Path.cwd().parent]
    if DRIVE_MOUNT_ROOT.exists():
        fallback_candidates.extend([
            DRIVE_MOUNT_ROOT / "MyDrive" / "SwellEx",
            DRIVE_MOUNT_ROOT / "MyDrive" / DRIVE_WORKSPACE_SUBDIR / "SwellEx",
        ])
    for candidate in fallback_candidates:
        if (candidate / "scripts" / "remote_workflow" / "packs.py").exists():
            print(f"⚠️ GitHub 克隆未成功，已回退使用挂载目录副本: {candidate.resolve()}")
            return candidate.resolve()
    raise FileNotFoundError(
        f"自动克隆仓库失败（可能仓库为私有）: {proc.stderr.strip()}\n"
        "请先将最新 SwellEx 仓库上传至 /content/drive/MyDrive/SwellEx 或使用带权限链接克隆。"
    )


PROJECT_ROOT = ensure_project_root()
reload_project_modules()
REPO_COMMIT, REPO_SUBJECT = get_git_commit_info(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if IN_COLAB:
    DRIVE_BASE = DRIVE_MOUNT_ROOT / "MyDrive" / DRIVE_WORKSPACE_SUBDIR
    LOCAL_DATASET_CACHE = Path("/content/local_datasets")
else:
    DRIVE_BASE = Path(os.environ.get("SWELLEX_DRIVE_SIM", PROJECT_ROOT / "outputs" / "drive_staging"))
    LOCAL_DATASET_CACHE = PROJECT_ROOT / "outputs" / "local_dataset_cache"

DRIVE_DATASET_ROOT = DRIVE_BASE / "datasets"
DRIVE_EXP_ROOT = DRIVE_BASE / "experiments"
MANIFEST_PATH = DRIVE_DATASET_ROOT / "dataset_manifest.json"

HAS_TORCH = importlib.util.find_spec("torch") is not None
CUDA_AVAILABLE = False
if HAS_TORCH:
    import torch
    CUDA_AVAILABLE = bool(torch.cuda.is_available())
    if CUDA_AVAILABLE:
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        print(f"✅ CUDA GPU 已就绪: {gpu_name} ({vram_gb:.1f} GiB), PyTorch {torch.__version__}")
    else:
        print("⚠️ 未检测到可用 CUDA GPU！在 Colab 中请切换 Runtime -> Change runtime type -> GPU。")
else:
    print("ℹ️ 当前本地解释器未安装 PyTorch（将以配置/历史结果分析模式运行；Colab GPU 环境自带 PyTorch）。")

MANIFEST = (
    json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    if MANIFEST_PATH.exists()
    else {"schema_version": 1, "presets": {}}
)
origin_prov = MANIFEST.get("origindata_source", {})
manifest_commit = MANIFEST.get("repo_commit")
print(f"PROJECT_ROOT       : {PROJECT_ROOT} (commit={REPO_COMMIT})")
print(f"MANIFEST_PATH      : {MANIFEST_PATH} (存在={MANIFEST_PATH.exists()}, 数据生成 commit={manifest_commit or '未知'})")
if origin_prov:
    print(f"原始数据溯源 (只读): FolderID={origin_prov.get('drive_folder_id')} ({origin_prov.get('resolution_mode')})")
elif not MANIFEST_PATH.exists():
    print(f"ℹ️ 尚未发现 dataset_manifest.json：请先在 CPU 运行时执行 01_data_processing_cpu.ipynb（已绑定 {ORIGINDATA_DRIVE_URL}）。")
print(f"可用预设数据集     : {list(MANIFEST.get('presets', {}).keys())}")
print(f"实验结果保存根目录 : {DRIVE_EXP_ROOT}")

## 1. `base_pack` 注册表速览

下方单元格调用仓库纯标准库模块（`scripts.remote_workflow.packs`），帮助在启动 GPU 训练前确认合法的 `METHOD`（`RBD` / `SCM`）与 `NETWORK`（`complex_CNN` / `ResNet18` / `ResNet50` / `GRNN`）。

In [ ]:
# Cell 2: 查看 base_pack 支持的特征路由/网络组合
from scripts.remote_workflow.packs import PACKS

print("=== 实验包注册表 (scripts/remote_workflow/packs.py) ===")
for name, pack in PACKS.items():
    print(f"- {name:10s}: methods={list(pack.methods)} | networks={list(pack.networks)}")

## 2. 神经网络与独立 GRNN 训练 / 预测运行器（GPU）

仅支持 `base_pack`（特征 `RBD` / `SCM`；网络 `complex_CNN` / `ResNet18` / `ResNet50` / `GRNN`）：
1. 自动从 `DRIVE_DATASET_ROOT` 将目标方法对应的 HDF5 目录复制到本地高速缓存 `LOCAL_DATASET_CACHE`；
2. 神经网络依次执行 `train`（保存最优验证集 `best.pt`）与 `predict`（输出测试集逐样本预测 CSV）；独立 `GRNN` 依次执行 `build`（5 折交叉验证选 `sigma`）与 `predict`；
3. 所有模型权重、历史记录和预测结果直接写入 Google Drive 的 `DRIVE_EXP_ROOT/<run_name>/`。

In [ ]:
# Cell 3: base_pack GPU 训练与测试预测（按需修改参数并将 RUN_TRAINING 设为 True）
from datetime import datetime
import subprocess

from scripts.network_training.catalog import entry_script as select_entry, resolve_selection

# --- 用户训练参数 ---
RUN_TRAINING = False               # 在 Colab GPU 就绪且模块一已生成数据后设为 True
DATASET_TAG = "train_before"       # 对应模块一写入 dataset_manifest.json 的 dataset_tag
METHOD = "SCM"                     # "RBD" | "SCM"
NETWORK = "ResNet18"               # "complex_CNN" | "ResNet18" | "ResNet50" | "GRNN"
EPOCHS = 100
BATCH_SIZE = 64
SEED = 2026
# --------------------


def stage_h5_from_drive(dataset_tag: str, method: str) -> Path:
    preset_info = MANIFEST.get("presets", {}).get(dataset_tag)
    if not preset_info:
        raise KeyError(
            f"dataset_manifest.json 中未找到 dataset_tag={dataset_tag!r}，请先在模块一生成并同步该预设。"
        )
    entry = preset_info["base_routes"][method]
    rel_dir = Path(entry["rel_dir"])
    src_dir = DRIVE_DATASET_ROOT / rel_dir
    dst_dir = LOCAL_DATASET_CACHE / rel_dir
    if dst_dir.exists():
        shutil.rmtree(dst_dir)
    dst_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src_dir, dst_dir)
    return dst_dir


selection = resolve_selection(METHOD, NETWORK)
entry_script = PROJECT_ROOT / select_entry(selection)
model_id = selection["model"]
prefix_args = ["--method", METHOD] if NETWORK == "GRNN" else []
print(f"目标入口脚本 : {entry_script.relative_to(PROJECT_ROOT)}")
print(f"解析模型标识 : {model_id}")

if not RUN_TRAINING:
    print("ℹ️ 当前 RUN_TRAINING=False（预览命令模式）。在 Colab GPU 准备好后将 RUN_TRAINING 设为 True 即可启动训练。")
else:
    if not CUDA_AVAILABLE:
        raise RuntimeError("拒绝静默回退到 CPU：请先将 Colab 运行时切换为 GPU！")
    local_data_dir = stage_h5_from_drive(DATASET_TAG, METHOD)
    stamp = datetime.now().strftime("%m%d_%H%M%S")
    exp_dir = DRIVE_EXP_ROOT / f"base_pack_{DATASET_TAG}_{METHOD}_{NETWORK}_{stamp}"
    exp_dir.mkdir(parents=True, exist_ok=True)
    print(f"📂 本地 SSD 数据缓存 : {local_data_dir}")
    print(f"💾 Google Drive 输出 : {exp_dir}")

    if NETWORK == "GRNN":
        cmds = [
            [sys.executable, "-B", str(entry_script), "build", *prefix_args,
             "--data", str(local_data_dir), "--device", "cuda",
             "--cv-folds", "5", "--batch-size", "1024", "--seed", str(SEED),
             "--output-dir", str(exp_dir)],
            [sys.executable, "-B", str(entry_script), "predict", *prefix_args,
             "--data", str(local_data_dir), "--device", "cuda",
             "--batch-size", "1024", "--output-dir", str(exp_dir)],
        ]
    else:
        cmds = [
            [sys.executable, "-B", str(entry_script), "train", "--method", METHOD,
             "--model", model_id, "--data", str(local_data_dir), "--device", "cuda",
             "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE),
             "--loss-space", "km", "--huber-beta", "0.5", "--seed", str(SEED),
             "--no-resume", "--output-dir", str(exp_dir)],
            [sys.executable, "-B", str(entry_script), "predict", "--method", METHOD,
             "--model", model_id, "--data", str(local_data_dir), "--device", "cuda",
             "--batch-size", str(BATCH_SIZE), "--output-dir", str(exp_dir)],
        ]

    for idx, cmd in enumerate(cmds, 1):
        print(f"\n>>> [{idx}/{len(cmds)}] 执行: {' '.join(cmd)}")
        subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
    print(f"\n🎉 训练与预测全部完成，结果已持久化至 Google Drive: {exp_dir}")

## 3. 实验结果汇总与对比可视化

自动扫描两类目录并汇总关键指标：
1. 本 Notebook 在 Google Drive `DRIVE_EXP_ROOT` 下新产出的全部实验目录（直接解析预测 CSV 计算 `MAE / RMSE / R²`，即使在未携带 `output/data/` 的 Colab 克隆环境中也可独立汇总与出图）；
2. 仓库本地 `output/data/` 中保留的 `base_pack` 基准运行（若存在则读取其 `reports/visualization/metrics.csv`，只展示 RBD/SCM 与 `complex_CNN` / `ResNet` / `GRNN` 行；这些运行含清理前的旧计划，ELM 与 `real_CNN` 行被忽略）。

In [ ]:
# Cell 4: 扫描 Google Drive 新实验结果与仓库 output/data 基准运行
import csv
import math


def compute_csv_metrics(pred_csv: Path) -> dict[str, float] | None:
    with pred_csv.open(encoding="utf-8") as fh:
        rows = list(csv.DictReader(fh))
    if not rows:
        return None
    pred_key = "pred_range_km" if "pred_range_km" in rows[0] else "predicted_range_km"
    preds = [float(r[pred_key]) for r in rows]
    trues = [float(r["true_range_km"]) for r in rows]
    n = len(rows)
    mae = sum(abs(p - t) for p, t in zip(preds, trues)) / n
    rmse = math.sqrt(sum((p - t) ** 2 for p, t in zip(preds, trues)) / n)
    bias = sum(p - t for p, t in zip(preds, trues)) / n
    mean_t = sum(trues) / n
    ss_tot = sum((t - mean_t) ** 2 for t in trues)
    r2 = 1.0 - (sum((p - t) ** 2 for p, t in zip(preds, trues)) / ss_tot) if ss_tot > 0 else float("nan")
    return {"count": n, "mae_km": mae, "rmse_km": rmse, "bias_km": bias, "r2": r2}


# 1) 扫描 Google Drive 实验目录中的新结果
DRIVE_RUN_METRICS: list[dict] = []
drive_runs = sorted(p for p in DRIVE_EXP_ROOT.iterdir() if p.is_dir()) if DRIVE_EXP_ROOT.exists() else []

print(f"=== Google Drive 实验产出目录 ({DRIVE_EXP_ROOT}) : 共 {len(drive_runs)} 个 ===")
for run_dir in drive_runs:
    pred_csvs = sorted(run_dir.rglob("*_predictions_*.csv"))
    if pred_csvs:
        m = compute_csv_metrics(pred_csvs[-1])
        if m:
            DRIVE_RUN_METRICS.append({"run": run_dir.name, **m})
            print(
                f"- {run_dir.name} | RMSE={m['rmse_km']:.4f} km | MAE={m['mae_km']:.4f} km "
                f"| bias={m['bias_km']:+.4f} km | R2={m['r2']:.4f} (N={m['count']})"
            )
    else:
        print(f"- {run_dir.name} (进行中或无测试集预测 CSV)")

# 2) 读取仓库 output/data 中 base_pack 基准运行（只保留 RBD/SCM 与当前网络；忽略 ELM 与 real_CNN 的旧行）
OUTPUT_DATA = PROJECT_ROOT / "output" / "data"
ARCHIVE_SUMMARY: dict[str, list] = {"base_pack": []}
for metrics_csv in sorted(OUTPUT_DATA.glob("*-base-*/reports/visualization/metrics.csv")):
    with metrics_csv.open(encoding="utf-8") as f:
        for r in csv.DictReader(f):
            if r["method"] in ("RBD", "SCM") and "real_cnn" not in r["model"]:
                ARCHIVE_SUMMARY["base_pack"].append({"run": metrics_csv.parents[2].name, **r})
ARCHIVE_SUMMARY["base_pack"].sort(key=lambda r: float(r["rmse_km"]))

if ARCHIVE_SUMMARY["base_pack"]:
    print("\n[基准参考] output/data 中 base_pack 运行 Top-5 测试集 RMSE:")
    for r in ARCHIVE_SUMMARY["base_pack"][:5]:
        print(f"  {r['run'][:40]:40s} | {r['method']:3s} | {r['model']:22s} | RMSE={float(r['rmse_km']):.4f} km | MAE={float(r['mae_km']):.4f} km")

In [ ]:
# Cell 5: 绘制实验对比看板（Google Drive 新产出实验与 output/data 基准运行）
has_plot_data = bool(DRIVE_RUN_METRICS or ARCHIVE_SUMMARY.get("base_pack"))
if importlib.util.find_spec("matplotlib") is None or not has_plot_data:
    print("ℹ️ 未检测到 matplotlib 或当前尚无已完成的实验/基准数据，跳过图形渲染（完成上方训练后重新运行本单元即可出图）。")
else:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

    ax = axes[0]
    if DRIVE_RUN_METRICS:
        top = sorted(DRIVE_RUN_METRICS, key=lambda r: r["rmse_km"])[:10]
        ax.barh([r["run"][:32] for r in top][::-1], [r["rmse_km"] for r in top][::-1], color="#2b6cb0")
        ax.set_title("Google Drive New Runs Test RMSE (km)")
    else:
        top = ARCHIVE_SUMMARY["base_pack"][:8]
        ax.barh([f"{r['method']}-{r['model']}" for r in top][::-1], [float(r["rmse_km"]) for r in top][::-1], color="#2b6cb0")
        ax.set_title("base_pack output/data Top-8 Test RMSE (km)")
    ax.set_xlabel("RMSE (km)")

    ax = axes[1]
    if DRIVE_RUN_METRICS:
        top = sorted(DRIVE_RUN_METRICS, key=lambda r: abs(r["bias_km"]))[:10]
        ax.barh([r["run"][:32] for r in top][::-1], [r["bias_km"] for r in top][::-1], color="#2f855a")
        ax.set_title("Google Drive New Runs Signed Bias (km)")
    else:
        top = ARCHIVE_SUMMARY["base_pack"][:8]
        ax.barh([f"{r['method']}-{r['model']}" for r in top][::-1], [float(r["bias_km"]) for r in top][::-1], color="#2f855a")
        ax.set_title("base_pack output/data Signed Bias (km)")
    ax.axvline(0, color="#555555", lw=1)
    ax.set_xlabel("Bias = prediction - truth (km)")

    if IN_COLAB or DRIVE_EXP_ROOT.exists():
        DRIVE_EXP_ROOT.mkdir(parents=True, exist_ok=True)
        fig_out = DRIVE_EXP_ROOT / "swellex_benchmark_overview.png"
        fig.savefig(fig_out, dpi=160)
        print(f"📊 对比图已保存至: {fig_out}")
    plt.show()